In [1]:
from src.retrievers.hybrid import HybridRetriever
from src.retrievers.bm25 import Bm25Retriever
from src.retrievers.dense import DenseRetriever
from src.embeddings import embed_texts, DEFAULT_BI_ENCODER
from src.data_utils import load_documents, chunker
from src.generate import generate
import faiss

documents = load_documents("/home/nick/new-speedrun/data/2038.pdf")
chunks = chunker(documents, chunk_size=301, chunk_overlap=30.1)
bm25_retriever = Bm25Retriever()
bm25_retriever.add_documents(chunks)


/home/nick/new-speedrun/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

query = 'What did Darren Walsh do for Work in 2027?'
pairs = [(query, doc.page_content) for doc in chunks]
scores = reranker.predict(pairs)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7358.67it/s]


In [9]:
ranked = sorted(zip(scores, chunks), reverse=True)

In [10]:

test_queries = [
    'Does Sterling Bronson have a mag?',
    'What did Darren Walsh do for Work in 2027?',
    'Which pub does Winston Reed regularly visit?',
    'What town is this novel set in?',
    'Who is Grace?',
    'Which chapter first introduces Cassie May?',
    'Which social or occupational class does Cassie belong to?',
    'What are Cats?',
    'what are transistors?',
    'does sterling own the transistors?',
    'what are the people who own the transistors referred to?'
]


def rag_answer(query: str, k: int = 40) -> str:
    if k < 1:
        raise ValueError("k must be at least 1")
    retrieved = bm25_retriever.retrieve(query, top_k=40)
    if not retrieved:
        return "I do not know"
    pairs = [(query, doc.page_content) for doc in retrieved]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, retrieved), key=lambda item: item[0], reverse=True)
    context_chunks = [doc.page_content for _, doc in ranked[:k]]
    return generate(query, context_chunks)



for query in test_queries:
    print(query)
    print(rag_answer(query))
    print()

Does Sterling Bronson have a mag?
Yes, Sterling Bronson has a Mag called the Clerk, which he made himself.

What did Darren Walsh do for Work in 2027?
In 2027, Darren Walsh was a scaffolder.

Which pub does Winston Reed regularly visit?
Winston Reed regularly visits a pub called The Crown.

What town is this novel set in?
The novel is set in the town of Brackton.

Who is Grace?
Grace is a character in the novel "2038" who appears to be a significant person in Reed's life. She is depicted as someone who knows Reed very well, understanding his moods and actions without needing explanations. Grace is often found on the sofa with her feet tucked under her, looking at Reed as though he is the best thing that happened all day. She is attentive, supportive, and seems to have an intuitive understanding of Reed, often responding in exactly the right way to his actions and words.

Which chapter first introduces Cassie May?
Cassie May is first introduced in Chapter Three.

Which social or occupat